In [0]:

base_path = (
    "abfss://globalmart@globalmartpiyush.dfs.core.windows.net/"
)

silver_customers_path = base_path + "silver/customers"
silver_products_path = base_path + "silver/products"
silver_stores_path = base_path + "silver/stores"
silver_regions_path = base_path + "silver/regions"
silver_orders_path = base_path + "silver/orders"
silver_order_items_path = base_path + "silver/order_items"

# Separate output path to protect the existing SCD dimension
gold_star_path = base_path + "gold/star_schema/"

print("GlobalMart Gold Star Schema initialized.")
print("Source: Silver Delta tables")
print("Target:", gold_star_path)


GlobalMart Gold Star Schema initialized.
Source: Silver Delta tables
Target: abfss://globalmart@globalmartpiyush.dfs.core.windows.net/gold/star_schema/


In [0]:

customers_df = (
    spark.read.format("delta")
    .load(silver_customers_path)
)

products_df = (
    spark.read.format("delta")
    .load(silver_products_path)
)

stores_df = (
    spark.read.format("delta")
    .load(silver_stores_path)
)

regions_df = (
    spark.read.format("delta")
    .load(silver_regions_path)
)

orders_df = (
    spark.read.format("delta")
    .load(silver_orders_path)
)

order_items_df = (
    spark.read.format("delta")
    .load(silver_order_items_path)
)

print("Customers:", customers_df.count())
print("Products:", products_df.count())
print("Stores:", stores_df.count())
print("Regions:", regions_df.count())
print("Orders:", orders_df.count())
print("Order items:", order_items_df.count())


Customers: 100
Products: 60
Stores: 15
Regions: 6
Orders: 504
Order items: 1298


In [0]:

from pyspark.sql.functions import col, row_number
from pyspark.sql.window import Window

# Generate a surrogate key for each customer
customer_window = Window.orderBy("customer_id")

dim_customer_df = (
    customers_df
    .dropDuplicates(["customer_id"])
    .withColumn(
        "customer_key",
        row_number().over(customer_window)
    )
    .select(
        "customer_key",
        "customer_id",
        "first_name",
        "last_name",
        "email",
        "city",
        "state",
        "country",
        "customer_segment",
        "created_date",
        "updated_date"
    )
)

# Save as a Gold Delta dimension
(
    dim_customer_df.write
    .format("delta")
    .mode("overwrite")
    .save(gold_star_path + "dim_customer")
)

print("Gold dim_customer created.")
print("Customer dimension rows:", dim_customer_df.count())

display(dim_customer_df.limit(10))


/databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/expressions.py:1155: UserWarning: WARN WindowExpression: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
  warnings.warn(


Gold dim_customer created.
Customer dimension rows: 100


customer_key,customer_id,first_name,last_name,email,city,state,country,customer_segment,created_date,updated_date
1,1,Arjun,Sharma,arjun.sharma1@example.com,Indore,Madhya Pradesh,India,Consumer,2024-10-08,2024-12-09
2,2,Rahul,Nair,rahul.nair2@example.com,Mumbai,Maharashtra,India,Premium,2025-07-12,2025-08-03
3,3,Pooja,Sharma,pooja.sharma3@example.com,Pune,Maharashtra,India,Consumer,2024-04-05,2024-05-30
4,4,Nisha,Khan,nisha.khan4@example.com,Pune,Maharashtra,India,Premium,2025-07-28,2025-09-16
5,5,Riya,Kulkarni,riya.kulkarni5@example.com,Delhi,Delhi,India,Corporate,2025-04-04,2025-09-01
6,6,Aarav,Reddy,aarav.reddy6@example.com,Lucknow,Uttar Pradesh,India,Corporate,2024-06-12,2024-12-07
7,7,Ananya,Joshi,ananya.joshi7@example.com,Nagpur,Maharashtra,India,Consumer,2024-08-08,2024-11-02
8,8,Aditya,Kulkarni,aditya.kulkarni8@example.com,Mumbai,Maharashtra,India,Premium,2025-01-02,2025-03-31
9,9,Neha,Reddy,neha.reddy9@example.com,Pune,Maharashtra,India,Consumer,2025-04-15,2025-08-30
10,10,Sneha,Patil,sneha.patil10@example.com,Kolkata,West Bengal,India,Premium,2024-10-27,2025-04-05


In [0]:

dim_customer_check_df = (
    spark.read
    .format("delta")
    .load(gold_star_path + "dim_customer")
)

print("Total rows:", dim_customer_check_df.count())

print(
    "Distinct customer IDs:",
    dim_customer_check_df.select("customer_id").distinct().count()
)

print(
    "Duplicate customer IDs:",
    dim_customer_check_df.groupBy("customer_id")
    .count()
    .filter(col("count") > 1)
    .count()
)


Total rows: 100
Distinct customer IDs: 100
Duplicate customer IDs: 0


In [0]:

from pyspark.sql.functions import col, row_number
from pyspark.sql.window import Window

# Generate a surrogate key for each product
product_window = Window.orderBy("product_id")

dim_product_df = (
    products_df
    .dropDuplicates(["product_id"])
    .withColumn(
        "product_key",
        row_number().over(product_window)
    )
    .select(
        "product_key",
        "product_id",
        "product_name",
        "category",
        "subcategory",
        "brand",
        "unit_price",
        "cost_price",
        "created_date",
        "updated_date"
    )
)

# Write the Gold dimension
(
    dim_product_df.write
    .format("delta")
    .mode("overwrite")
    .save(gold_star_path + "dim_product")
)

print("Gold dim_product created.")
print("Product dimension rows:", dim_product_df.count())

display(dim_product_df.limit(10))


/databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/expressions.py:1155: UserWarning: WARN WindowExpression: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
  warnings.warn(


Gold dim_product created.
Product dimension rows: 60


product_key,product_id,product_name,category,subcategory,brand,unit_price,cost_price,created_date,updated_date
1,1001,Apex Mobile 1001,Electronics,Mobile,Zenith,37496,27030.64,2024-01-01,2025-01-01
2,1002,Nova Mobile 1002,Electronics,Mobile,Vertex,44233,32363.29,2024-01-01,2025-01-01
3,1003,Vertex Mobile 1003,Electronics,Mobile,Vertex,29744,16555.36,2024-01-01,2025-01-01
4,1004,Prime Laptop 1004,Electronics,Laptop,Apex,32392,25247.27,2024-01-01,2025-01-01
5,1005,Prime Laptop 1005,Electronics,Laptop,Prime,9845,7770.23,2024-01-01,2025-01-01
6,1006,Vertex Laptop 1006,Electronics,Laptop,Zenith,75754,55907.64,2024-01-01,2025-01-01
7,1007,Prime Audio 1007,Electronics,Audio,Vertex,65099,38398.27,2024-01-01,2025-01-01
8,1008,Nova Audio 1008,Electronics,Audio,Prime,1026,624.92,2024-01-01,2025-01-01
9,1009,Prime Audio 1009,Electronics,Audio,Nova,68189,47766.36,2024-01-01,2025-01-01
10,1010,Prime Accessories 1010,Electronics,Accessories,Vertex,16206,12419.99,2024-01-01,2025-01-01


In [0]:

dim_product_check_df = (
    spark.read
    .format("delta")
    .load(gold_star_path + "dim_product")
)

print("Total products:", dim_product_check_df.count())

print(
    "Distinct product IDs:",
    dim_product_check_df.select("product_id").distinct().count()
)

print(
    "Duplicate product IDs:",
    dim_product_check_df.groupBy("product_id")
    .count()
    .filter(col("count") > 1)
    .count()
)

print(
    "Invalid prices:",
    dim_product_check_df.filter(
        (col("unit_price") <= 0) |
        (col("cost_price") <= 0) |
        (col("cost_price") > col("unit_price"))
    ).count()
)


Total products: 60
Distinct product IDs: 60
Duplicate product IDs: 0
Invalid prices: 0


In [0]:
from pyspark.sql.functions import col, row_number
from pyspark.sql.window import Window

# 1. Join stores with regions
store_joined_df = (
    stores_df.alias("s")
    .join(
        regions_df.alias("r"),
        col("s.region_id") == col("r.region_id"),
        "left"
    )
    .select(
        col("s.store_id").alias("store_id"),
        col("s.store_name").alias("store_name"),
        col("s.city").alias("city"),
        col("s.state").alias("state"),
        col("s.country").alias("country"),
        col("s.region_id").alias("region_id"),
        col("r.region_name").alias("region_name")
    )
    .dropDuplicates(["store_id"])
)

# 2. Generate a surrogate key
store_window = Window.orderBy("store_id")

dim_store_df = store_joined_df.withColumn(
    "store_key",
    row_number().over(store_window)
).select(
    "store_key",
    "store_id",
    "store_name",
    "city",
    "state",
    "country",
    "region_id",
    "region_name"
)

# 3. Save the Gold dimension
(
    dim_store_df.write
    .format("delta")
    .mode("overwrite")
    .save(gold_star_path + "dim_store")
)

# 4. Validate
print("Gold dim_store created.")
print("Store dimension rows:", dim_store_df.count())

display(dim_store_df.limit(10))

/databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/expressions.py:1155: UserWarning: WARN WindowExpression: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
  warnings.warn(


Gold dim_store created.
Store dimension rows: 15


store_key,store_id,store_name,city,state,country,region_id,region_name
1,101,GlobalMart Pune,Pune,Maharashtra,India,3,West
2,102,GlobalMart Mumbai,Mumbai,Maharashtra,India,3,West
3,103,GlobalMart Nagpur,Nagpur,Maharashtra,India,5,Central
4,104,GlobalMart Delhi,Delhi,Delhi,India,1,North
5,105,GlobalMart Jaipur,Jaipur,Rajasthan,India,1,North
6,106,GlobalMart Chennai,Chennai,Tamil Nadu,India,2,South
7,107,GlobalMart Bengaluru,Bengaluru,Karnataka,India,2,South
8,108,GlobalMart Hyderabad,Hyderabad,Telangana,India,2,South
9,109,GlobalMart Kolkata,Kolkata,West Bengal,India,4,East
10,110,GlobalMart Bhubaneswar,Bhubaneswar,Odisha,India,4,East


In [0]:

dim_store_check_df = (
    spark.read
    .format("delta")
    .load(gold_star_path + "dim_store")
)

print("Total stores:", dim_store_check_df.count())

print(
    "Distinct store IDs:",
    dim_store_check_df.select("store_id").distinct().count()
)

print(
    "Duplicate store IDs:",
    dim_store_check_df.groupBy("store_id")
    .count()
    .filter(col("count") > 1)
    .count()
)

print(
    "Missing region names:",
    dim_store_check_df.filter(col("region_name").isNull()).count()
)

display(
    dim_store_check_df.select(
        "store_id",
        "store_name",
        "city",
        "country",
        "region_name"
    )
)


Total stores: 15
Distinct store IDs: 15
Duplicate store IDs: 0
Missing region names: 0


store_id,store_name,city,country,region_name
101,GlobalMart Pune,Pune,India,West
102,GlobalMart Mumbai,Mumbai,India,West
103,GlobalMart Nagpur,Nagpur,India,Central
104,GlobalMart Delhi,Delhi,India,North
105,GlobalMart Jaipur,Jaipur,India,North
106,GlobalMart Chennai,Chennai,India,South
107,GlobalMart Bengaluru,Bengaluru,India,South
108,GlobalMart Hyderabad,Hyderabad,India,South
109,GlobalMart Kolkata,Kolkata,India,East
110,GlobalMart Bhubaneswar,Bhubaneswar,India,East


In [0]:

from pyspark.sql.functions import (
    col,
    to_date,
    date_format,
    year,
    quarter,
    month,
    dayofmonth,
    dayofweek,
    dayofyear,
    weekofyear,
    last_day
)

# Get distinct order dates from Silver Orders
order_dates_df = (
    orders_df
    .select(to_date("order_date").alias("date"))
    .filter(col("date").isNotNull())
    .distinct()
)

# Build the Gold date dimension
dim_date_df = (
    order_dates_df
    .withColumn(
        "date_key",
        date_format("date", "yyyyMMdd").cast("int")
    )
    .withColumn("year", year("date"))
    .withColumn("quarter", quarter("date"))
    .withColumn("month", month("date"))
    .withColumn("month_name", date_format("date", "MMMM"))
    .withColumn("day", dayofmonth("date"))
    .withColumn("day_of_week", date_format("date", "EEEE"))
    .withColumn("day_of_week_number", dayofweek("date"))
    .withColumn("day_of_year", dayofyear("date"))
    .withColumn("week_of_year", weekofyear("date"))
    .withColumn("month_end_date", last_day("date"))
    .select(
        "date_key",
        "date",
        "year",
        "quarter",
        "month",
        "month_name",
        "day",
        "day_of_week",
        "day_of_week_number",
        "day_of_year",
        "week_of_year",
        "month_end_date"
    )
)

# Save as Gold Delta
(
    dim_date_df.write
    .format("delta")
    .mode("overwrite")
    .save(gold_star_path + "dim_date")
)

print("Gold dim_date created.")
print("Date dimension rows:", dim_date_df.count())

display(
    dim_date_df.orderBy("date").limit(10)
)


Gold dim_date created.
Date dimension rows: 266


date_key,date,year,quarter,month,month_name,day,day_of_week,day_of_week_number,day_of_year,week_of_year,month_end_date
20250101,2025-01-01,2025,1,1,January,1,Wednesday,4,1,1,2025-01-31
20250102,2025-01-02,2025,1,1,January,2,Thursday,5,2,1,2025-01-31
20250103,2025-01-03,2025,1,1,January,3,Friday,6,3,1,2025-01-31
20250107,2025-01-07,2025,1,1,January,7,Tuesday,3,7,2,2025-01-31
20250108,2025-01-08,2025,1,1,January,8,Wednesday,4,8,2,2025-01-31
20250109,2025-01-09,2025,1,1,January,9,Thursday,5,9,2,2025-01-31
20250112,2025-01-12,2025,1,1,January,12,Sunday,1,12,2,2025-01-31
20250116,2025-01-16,2025,1,1,January,16,Thursday,5,16,3,2025-01-31
20250117,2025-01-17,2025,1,1,January,17,Friday,6,17,3,2025-01-31
20250118,2025-01-18,2025,1,1,January,18,Saturday,7,18,3,2025-01-31


In [0]:

dim_date_check_df = (
    spark.read
    .format("delta")
    .load(gold_star_path + "dim_date")
)

print("Total date rows:", dim_date_check_df.count())

print(
    "Duplicate date keys:",
    dim_date_check_df.groupBy("date_key")
    .count()
    .filter(col("count") > 1)
    .count()
)

print(
    "Missing date keys:",
    dim_date_check_df.filter(col("date_key").isNull()).count()
)

display(
    dim_date_check_df
    .select(
        "date_key",
        "date",
        "year",
        "quarter",
        "month",
        "month_name",
        "day_of_week"
    )
    .orderBy("date")
)


Total date rows: 266
Duplicate date keys: 0
Missing date keys: 0


date_key,date,year,quarter,month,month_name,day_of_week
20250101,2025-01-01,2025,1,1,January,Wednesday
20250102,2025-01-02,2025,1,1,January,Thursday
20250103,2025-01-03,2025,1,1,January,Friday
20250107,2025-01-07,2025,1,1,January,Tuesday
20250108,2025-01-08,2025,1,1,January,Wednesday
20250109,2025-01-09,2025,1,1,January,Thursday
20250112,2025-01-12,2025,1,1,January,Sunday
20250116,2025-01-16,2025,1,1,January,Thursday
20250117,2025-01-17,2025,1,1,January,Friday
20250118,2025-01-18,2025,1,1,January,Saturday


In [0]:

from pyspark.sql.functions import (
    col,
    to_date,
    coalesce,
    lit,
    sum as spark_sum
)

# ============================================================
# 1. READ GOLD DIMENSION TABLES
# ============================================================

dim_customer_df = (
    spark.read.format("delta")
    .load(gold_star_path + "dim_customer")
)

dim_product_df = (
    spark.read.format("delta")
    .load(gold_star_path + "dim_product")
)

dim_store_df = (
    spark.read.format("delta")
    .load(gold_star_path + "dim_store")
)

dim_date_df = (
    spark.read.format("delta")
    .load(gold_star_path + "dim_date")
)


# ============================================================
# 2. PREPARE CUSTOMER DIMENSION LOOKUP
# ============================================================
# Your current dim_customer does not contain is_current.
# Use its existing customer_id and customer_key columns.

customer_lookup_df = (
    dim_customer_df
    .select(
        "customer_id",
        "customer_key"
    )
)


# ============================================================
# 3. PREPARE DATE DIMENSION LOOKUP
# ============================================================

date_lookup_df = (
    dim_date_df
    .select(
        to_date(col("date")).alias("calendar_date"),
        col("date_key")
    )
)


# ============================================================
# 4. BUILD SALES FACT TABLE
# Grain: one row per order item
# ============================================================

fact_sales_df = (
    order_items_df.alias("oi")

    # Join order items to their order headers
    .join(
        orders_df.alias("o"),
        col("oi.order_id") == col("o.order_id"),
        "inner"
    )

    # Resolve customer surrogate key
    .join(
        customer_lookup_df.alias("c"),
        col("o.customer_id") == col("c.customer_id"),
        "left"
    )

    # Resolve product surrogate key
    .join(
        dim_product_df.alias("p"),
        col("oi.product_id") == col("p.product_id"),
        "left"
    )

    # Resolve store surrogate key
    .join(
        dim_store_df.alias("s"),
        col("o.store_id") == col("s.store_id"),
        "left"
    )

    # Resolve date surrogate key
    .join(
        date_lookup_df.alias("d"),
        to_date(col("o.order_date")) == col("d.calendar_date"),
        "left"
    )

    # Select fact columns and calculate sales measures
    .select(
        col("oi.order_item_id").alias("sales_key_source_id"),
        col("oi.order_id").alias("order_id"),

        col("c.customer_key").alias("customer_key"),
        col("p.product_key").alias("product_key"),
        col("s.store_key").alias("store_key"),
        col("d.date_key").alias("date_key"),

        col("oi.quantity").alias("quantity"),
        col("oi.unit_price").alias("unit_price"),

        coalesce(
            col("oi.discount"),
            lit(0)
        ).alias("discount"),

        (
            col("oi.quantity") * col("oi.unit_price")
        ).alias("gross_sales"),

        (
            (col("oi.quantity") * col("oi.unit_price"))
            - coalesce(col("oi.discount"), lit(0))
        ).alias("net_sales")
    )
)


# ============================================================
# 5. VALIDATE FACT TABLE BEFORE WRITING
# ============================================================

print("Fact table schema:")
fact_sales_df.printSchema()

print("Total fact rows:", fact_sales_df.count())

# Check for duplicate order-item records
print("Duplicate order-item records:")

(
    fact_sales_df
    .groupBy("sales_key_source_id")
    .count()
    .filter(col("count") > 1)
    .show()
)

# Check missing dimension keys
print("Missing dimension key counts:")

fact_sales_df.select(
    spark_sum(
        col("customer_key").isNull().cast("int")
    ).alias("missing_customer_keys"),

    spark_sum(
        col("product_key").isNull().cast("int")
    ).alias("missing_product_keys"),

    spark_sum(
        col("store_key").isNull().cast("int")
    ).alias("missing_store_keys"),

    spark_sum(
        col("date_key").isNull().cast("int")
    ).alias("missing_date_keys")
).show()

# Preview the data
display(fact_sales_df.limit(10))


# ============================================================
# 6. WRITE GOLD FACT TABLE
# ============================================================
# Full refresh for the initial star-schema implementation.

(
    fact_sales_df.write
    .format("delta")
    .mode("overwrite")
    .save(gold_star_path + "fact_sales")
)

print("Gold fact_sales created successfully.")
print(
    "Final fact rows:",
    spark.read.format("delta")
    .load(gold_star_path + "fact_sales")
    .count()
)


Fact table schema:
root
 |-- sales_key_source_id: integer (nullable = true)
 |-- order_id: integer (nullable = true)
 |-- customer_key: integer (nullable = true)
 |-- product_key: integer (nullable = true)
 |-- store_key: integer (nullable = true)
 |-- date_key: integer (nullable = true)
 |-- quantity: integer (nullable = true)
 |-- unit_price: double (nullable = true)
 |-- discount: double (nullable = false)
 |-- gross_sales: double (nullable = true)
 |-- net_sales: double (nullable = true)

Total fact rows: 1298
Duplicate order-item records:
+-------------------+-----+
|sales_key_source_id|count|
+-------------------+-----+
+-------------------+-----+

Missing dimension key counts:
+---------------------+--------------------+------------------+-----------------+
|missing_customer_keys|missing_product_keys|missing_store_keys|missing_date_keys|
+---------------------+--------------------+------------------+-----------------+
|                    0|                   0|                 

sales_key_source_id,order_id,customer_key,product_key,store_key,date_key,quantity,unit_price,discount,gross_sales,net_sales
12,5006,96,2,9,20250611,3,44233.0,0.05,132699.0,132698.95
18,5009,69,48,3,20250108,4,48871.0,0.15,195484.0,195483.85
38,5017,13,42,13,20251109,2,15312.0,0.15,30624.0,30623.85
67,5027,74,56,10,20250428,5,49974.0,0.15,249870.0,249869.85
70,5030,43,27,6,20250628,1,57425.0,0.0,57425.0,57425.0
93,5039,35,59,7,20250918,4,36511.0,0.0,146044.0,146044.0
161,5063,51,24,5,20250112,1,46405.0,0.0,46405.0,46405.0
186,5073,96,52,11,20250708,2,35700.0,0.0,71400.0,71400.0
190,5074,14,46,11,20251123,3,39829.0,0.05,119487.0,119486.95
218,5087,85,39,14,20250331,1,62370.0,0.15,62370.0,62369.85


Gold fact_sales created successfully.
Final fact rows: 1298


In [0]:

fact_check_df = (
    spark.read.format("delta")
    .load(gold_star_path + "fact_sales")
)

print("Total fact rows:", fact_check_df.count())

print(
    "Distinct order items:",
    fact_check_df.select("order_id").distinct().count()
)

print(
    "Duplicate order items:",
    fact_check_df.groupBy("order_id")
    .count()
    .filter(col("count") > 1)
    .count()
)

print(
    "Missing customer keys:",
    fact_check_df.filter(col("customer_key").isNull()).count()
)

print(
    "Missing product keys:",
    fact_check_df.filter(col("product_key").isNull()).count()
)

print(
    "Missing store keys:",
    fact_check_df.filter(col("store_key").isNull()).count()
)

print(
    "Missing date keys:",
    fact_check_df.filter(col("date_key").isNull()).count()
)


Total fact rows: 1298
Distinct order items: 500
Duplicate order items: 387
Missing customer keys: 0
Missing product keys: 0
Missing store keys: 0
Missing date keys: 0


In [0]:

from pyspark.sql.functions import col, count

# 1. Check uniqueness in the original Silver order items
duplicate_source_items = (
    order_items_df
    .groupBy("order_item_id")
    .agg(count("*").alias("row_count"))
    .filter(col("row_count") > 1)
)

print(
    "Duplicate IDs in Silver order_items:",
    duplicate_source_items.count()
)

display(duplicate_source_items.orderBy(col("row_count").desc()))

# 2. Check how many rows exist per order ID in the source
order_item_counts = (
    order_items_df
    .groupBy("order_id")
    .agg(count("*").alias("item_count"))
)

print("Distinct orders in order_items:", order_item_counts.count())

# 3. Check whether Silver Orders has duplicate order IDs
duplicate_orders = (
    orders_df
    .groupBy("order_id")
    .agg(count("*").alias("row_count"))
    .filter(col("row_count") > 1)
)

print("Duplicate IDs in Silver orders:", duplicate_orders.count())

display(duplicate_orders)


Duplicate IDs in Silver order_items: 0


order_item_id,row_count


Distinct orders in order_items: 500
Duplicate IDs in Silver orders: 0


order_id,row_count


In [0]:
print("Silver order_items rows:", order_items_df.count())
print(
    "Distinct order_item_id:",
    order_items_df.select("order_item_id").distinct().count()
)

print("Silver orders rows:", orders_df.count())
print(
    "Distinct order_id:",
    orders_df.select("order_id").distinct().count()
)

Silver order_items rows: 1298
Distinct order_item_id: 1298
Silver orders rows: 504
Distinct order_id: 504


In [0]:
from pyspark.sql.functions import col, count

print("Source order-item rows:", order_items_df.count())
print(
    "Distinct source order-item IDs:",
    order_items_df.select("order_item_id").distinct().count()
)

print("Source order rows:", orders_df.count())
print(
    "Distinct source order IDs:",
    orders_df.select("order_id").distinct().count()
)

print("Fact rows:", fact_sales_df.count())
print(
    "Distinct fact order-item IDs:",
    fact_sales_df.select("sales_key_source_id").distinct().count()
)

Source order-item rows: 1298
Distinct source order-item IDs: 1298
Source order rows: 504
Distinct source order IDs: 504
Fact rows: 1298
Distinct fact order-item IDs: 1298


In [0]:
from pyspark.sql.functions import col, count, sum as spark_sum

# 1. Confirm one fact row per order item
fact_sales_df.groupBy("sales_key_source_id") \
    .agg(count("*").alias("row_count")) \
    .filter(col("row_count") > 1) \
    .show()

# 2. Check missing dimension keys
fact_sales_df.select(
    spark_sum(col("customer_key").isNull().cast("int"))
        .alias("missing_customer_keys"),
    spark_sum(col("product_key").isNull().cast("int"))
        .alias("missing_product_keys"),
    spark_sum(col("store_key").isNull().cast("int"))
        .alias("missing_store_keys"),
    spark_sum(col("date_key").isNull().cast("int"))
        .alias("missing_date_keys")
).show()

# 3. Check negative quantities or prices
fact_sales_df.filter(
    (col("quantity") <= 0) | (col("unit_price") < 0)
).show()

# 4. Summarize sales
fact_sales_df.select(
    spark_sum("gross_sales").alias("total_gross_sales"),
    spark_sum("net_sales").alias("total_net_sales"),
    spark_sum("quantity").alias("total_quantity")
).show()

+-------------------+---------+
|sales_key_source_id|row_count|
+-------------------+---------+
+-------------------+---------+

+---------------------+--------------------+------------------+-----------------+
|missing_customer_keys|missing_product_keys|missing_store_keys|missing_date_keys|
+---------------------+--------------------+------------------+-----------------+
|                    0|                   0|                 0|                0|
+---------------------+--------------------+------------------+-----------------+

+-------------------+--------+------------+-----------+---------+--------+--------+----------+--------+-----------+---------+
|sales_key_source_id|order_id|customer_key|product_key|store_key|date_key|quantity|unit_price|discount|gross_sales|net_sales|
+-------------------+--------+------------+-----------+---------+--------+--------+----------+--------+-----------+---------+
+-------------------+--------+------------+-----------+---------+--------+--------

In [0]:

from pyspark.sql.functions import col, count

# Load all Gold star-schema tables
dim_customer = spark.read.format("delta").load(gold_star_path + "dim_customer")
dim_product  = spark.read.format("delta").load(gold_star_path + "dim_product")
dim_store    = spark.read.format("delta").load(gold_star_path + "dim_store")
dim_date     = spark.read.format("delta").load(gold_star_path + "dim_date")
fact_sales   = spark.read.format("delta").load(gold_star_path + "fact_sales")

# Row counts
print("GOLD STAR SCHEMA ROW COUNTS")
print("dim_customer:", dim_customer.count())
print("dim_product:", dim_product.count())
print("dim_store:", dim_store.count())
print("dim_date:", dim_date.count())
print("fact_sales:", fact_sales.count())

# Check dimension key uniqueness
print("\nDUPLICATE DIMENSION KEYS")

for df, key in [
    (dim_customer, "customer_key"),
    (dim_product, "product_key"),
    (dim_store, "store_key"),
    (dim_date, "date_key")
]:
    duplicates = (
        df.groupBy(key)
          .agg(count("*").alias("row_count"))
          .filter(col("row_count") > 1)
          .count()
    )
    print(f"{key}: {duplicates} duplicate keys")

# Check fact-table foreign keys
print("\nFACT FOREIGN KEY NULLS")
fact_sales.selectExpr(
    "sum(case when customer_key is null then 1 else 0 end) as missing_customer_keys",
    "sum(case when product_key is null then 1 else 0 end) as missing_product_keys",
    "sum(case when store_key is null then 1 else 0 end) as missing_store_keys",
    "sum(case when date_key is null then 1 else 0 end) as missing_date_keys"
).show()


GOLD STAR SCHEMA ROW COUNTS
dim_customer: 100
dim_product: 60
dim_store: 15
dim_date: 266
fact_sales: 1298

DUPLICATE DIMENSION KEYS
customer_key: 0 duplicate keys
product_key: 0 duplicate keys
store_key: 0 duplicate keys
date_key: 0 duplicate keys

FACT FOREIGN KEY NULLS
+---------------------+--------------------+------------------+-----------------+
|missing_customer_keys|missing_product_keys|missing_store_keys|missing_date_keys|
+---------------------+--------------------+------------------+-----------------+
|                    0|                   0|                 0|                0|
+---------------------+--------------------+------------------+-----------------+



In [0]:

fact_sales_df.explain("formatted")


== Physical Plan ==
AdaptiveSparkPlan (36)
+- == Initial Plan ==
   PhotonResultStage (35)
   +- PhotonColumnarToRow (34)
      +- PhotonProject (33)
         +- PhotonBroadcastHashJoin LeftOuter (32)
            :- PhotonProject (26)
            :  +- PhotonBroadcastHashJoin LeftOuter (25)
            :     :- PhotonProject (20)
            :     :  +- PhotonBroadcastHashJoin LeftOuter (19)
            :     :     :- PhotonProject (14)
            :     :     :  +- PhotonBroadcastHashJoin LeftOuter (13)
            :     :     :     :- PhotonProject (7)
            :     :     :     :  +- PhotonBroadcastHashJoin Inner (6)
            :     :     :     :     :- PhotonScan parquet  (1)
            :     :     :     :     +- PhotonShuffleExchangeSource (5)
            :     :     :     :        +- PhotonShuffleMapStage (4)
            :     :     :     :           +- PhotonShuffleExchangeSink (3)
            :     :     :     :              +- PhotonScan parquet  (2)
            :     : 

In [0]:

import time

# Warm up the environment and record a baseline
start_time = time.perf_counter()

fact_sales_df.count()

baseline_seconds = time.perf_counter() - start_time

print(f"Fact rows: {fact_sales_df.count()}")
print(f"Baseline execution time: {baseline_seconds:.3f} seconds")


Fact rows: 1298
Baseline execution time: 0.796 seconds


In [0]:

import time
from pyspark.sql.functions import col

# 1. Baseline: verify current Gold fact table
fact_path = gold_star_path + "fact_sales"

start = time.perf_counter()
fact_sales_df.count()
baseline_seconds = time.perf_counter() - start

print(f"Fact count time: {baseline_seconds:.3f} seconds")

# 2. Repartition experiment
print("\nPARTITIONING")

original_partitions = fact_sales_df.rdd.getNumPartitions()
print("Original partitions:", original_partitions)

repartitioned_df = fact_sales_df.repartition(4, "date_key")
print("Repartitioned partitions:",
      repartitioned_df.rdd.getNumPartitions())

coalesced_df = fact_sales_df.coalesce(1)
print("Coalesced partitions:",
      coalesced_df.rdd.getNumPartitions())

# Verify neither operation changes the number of rows
print("Repartitioned rows:", repartitioned_df.count())
print("Coalesced rows:", coalesced_df.count())

# 3. Inspect relevant Spark settings
print("\nSPARK SETTINGS")

for setting in [
    "spark.sql.adaptive.enabled",
    "spark.sql.shuffle.partitions",
    "spark.sql.autoBroadcastJoinThreshold"
]:
    try:
        print(setting, "=", spark.conf.get(setting))
    except Exception:
        print(setting, "= unavailable")

# 4. Optional Delta file compaction
print("\nDELTA OPTIMIZATION")

try:
    spark.sql(f"OPTIMIZE delta.`{fact_path}`")
    print("OPTIMIZE completed.")
except Exception as exc:
    print("OPTIMIZE unavailable or unsuccessful:", str(exc)[:300])

# 5. Final validation
saved_fact_df = (
    spark.read.format("delta")
    .load(fact_path)
)

final_rows = saved_fact_df.count()
distinct_items = (
    saved_fact_df.select("sales_key_source_id")
    .distinct()
    .count()
)

print("\nFINAL VALIDATION")
print("Fact rows:", final_rows)
print("Distinct item IDs:", distinct_items)

assert final_rows == 1298, "Unexpected fact row count"
assert distinct_items == final_rows, "Duplicate item IDs found"

print("Validation passed.")


Fact count time: 0.759 seconds

PARTITIONING


---------------------------------------------------------------------------
PySparkNotImplementedError                Traceback (most recent call last)
File <command-8762438295338473>, line 17
     14 # 2. Repartition experiment
     15 print("\nPARTITIONING")
---> 17 original_partitions = fact_sales_df.rdd.getNumPartitions()
     18 print("Original partitions:", original_partitions)
     20 repartitioned_df = fact_sales_df.repartition(4, "date_key")

File /databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/dataframe.py:2665, in DataFrame.rdd(self)
   2663 @property
   2664 def rdd(self) -> "RDD[Row]":
-> 2665     raise PySparkNotImplementedError(
   2666         errorClass="RDD_NOT_SUPPORTED",
   2667         messageParameters={},
   2668     )

PySparkNotImplementedError: [RDD_NOT_SUPPORTED] Using custom code using PySpark RDDs is not allowed on serverless compute. We suggest using mapInPandas or mapInArrow for the most common use cases, or switch to Dedicated access 

In [0]:
from pyspark.sql.functions import col

# Original DataFrame
print("Original rows:", fact_sales_df.count())

# Repartition into 4 partitions by date_key
repartitioned_df = fact_sales_df.repartition(4, col("date_key"))

# Reduce to 1 partition
coalesced_df = fact_sales_df.coalesce(1)

# Verify that repartitioning and coalescing preserve data
print("Repartitioned rows:", repartitioned_df.count())
print("Coalesced rows:", coalesced_df.count())

Original rows: 1298
Repartitioned rows: 1298
Coalesced rows: 1298


In [0]:
from pyspark.sql import functions as F

fact_path = (
    "abfss://globalmart@globalmartpiyush.dfs.core.windows.net/"
    "gold/star_schema/fact_sales"
)

fact_df = spark.read.format("delta").load(fact_path)

print("Fact row count:", fact_df.count())
print(
    "Distinct order-item IDs:",
    fact_df.select("sales_key_source_id").distinct().count()
)

print("AQE enabled:",
      spark.conf.get("spark.sql.adaptive.enabled", "not set"))

print("Auto broadcast threshold:",
      spark.conf.get("spark.sql.autoBroadcastJoinThreshold", "not set"))

fact_df.explain("formatted")

Fact row count: 1298
Distinct order-item IDs: 1298


---------------------------------------------------------------------------
AnalysisException                         Traceback (most recent call last)
File <command-8762438295338475>, line 17
     10 print("Fact row count:", fact_df.count())
     11 print(
     12     "Distinct order-item IDs:",
     13     fact_df.select("sales_key_source_id").distinct().count()
     14 )
     16 print("AQE enabled:",
---> 17       spark.conf.get("spark.sql.adaptive.enabled", "not set"))
     19 print("Auto broadcast threshold:",
     20       spark.conf.get("spark.sql.autoBroadcastJoinThreshold", "not set"))
     22 fact_df.explain("formatted")

File /databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/conf.py:87, in RuntimeConf.get(self, key, default)
     83     op_get_with_default = proto.ConfigRequest.GetWithDefault(
     84         pairs=[proto.KeyValue(key=key, value=cast(Optional[str], default))]
     85     )
     86     operation = proto.ConfigRequest.Operation(get_with_defau

In [0]:
from pyspark.sql import functions as F

base_path = (
    "abfss://globalmart@globalmartpiyush.dfs.core.windows.net/"
    "gold/star_schema/"
)

tables = [
    "dim_customer",
    "dim_product",
    "dim_store",
    "dim_date",
    "fact_sales"
]

for table_name in tables:
    df = spark.read.format("delta").load(base_path + table_name)

    print(f"{table_name}: {df.count()} rows")

dim_customer: 100 rows
dim_product: 60 rows
dim_store: 15 rows
dim_date: 266 rows
fact_sales: 1298 rows


In [0]:
export_base = "dbfs:/FileStore/globalmart_powerbi/"

for table_name in tables:
    df = spark.read.format("delta").load(base_path + table_name)

    (
        df.coalesce(1)
          .write
          .mode("overwrite")
          .option("header", True)
          .csv('/tmp/' + table_name)
    )

    print(f"Exported: {table_name}")

---------------------------------------------------------------------------
UnsupportedOperationException             Traceback (most recent call last)
File <command-7536963277210771>, line 11
      3 for table_name in tables:
      4     df = spark.read.format("delta").load(base_path + table_name)
      6     (
      7         df.coalesce(1)
      8           .write
      9           .mode("overwrite")
     10           .option("header", True)
---> 11           .csv('/tmp/' + table_name)
     12     )
     14     print(f"Exported: {table_name}")

File /databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/readwriter.py:922, in DataFrameWriter.csv(self, path, mode, compression, sep, quote, escape, header, nullValue, escapeQuotes, quoteAll, dateFormat, timestampFormat, ignoreLeadingWhiteSpace, ignoreTrailingWhiteSpace, charToEscapeQuoteEscaping, encoding, emptyValue, lineSep)
    903 self.mode(mode)
    904 self._set_opts(
    905     compression=compression,
    906     se

In [0]:
# Check available catalogs
display(spark.sql("SHOW CATALOGS"))
# Check schemas in the current catalog
display(spark.sql("SHOW SCHEMAS IN main"))

catalog
dbw_globalmart
samples
system


---------------------------------------------------------------------------
AnalysisException                         Traceback (most recent call last)
File <command-7536963277210772>, line 4
      2 display(spark.sql("SHOW CATALOGS"))
      3 # Check schemas in the current catalog
----> 4 display(spark.sql("SHOW SCHEMAS IN main"))

File /databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/session.py:919, in SparkSession.sql(self, sqlQuery, args, **kwargs)
    916         _views.append(SubqueryAlias(df._plan, name))
    918 cmd = SQL(sqlQuery, _args, _named_args, _views)
--> 919 data, properties, ei = self.client.execute_command(cmd.command(self._client))
    920 if "sql_command_result" in properties:
    921     df = DataFrame(CachedRelation(properties["sql_command_result"]), self)

File /databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/client/core.py:1812, in SparkConnectClient.execute_command(self, command, observations, extra_request_metadata)
   18

In [0]:
display(spark.sql("SHOW SCHEMAS IN dbw_globalmart"))

display(
    spark.sql("SHOW VOLUMES IN dbw_globalmart.your_schema")
)

databaseName
default
information_schema


---------------------------------------------------------------------------
AnalysisException                         Traceback (most recent call last)
File <command-7536963277210773>, line 4
      1 display(spark.sql("SHOW SCHEMAS IN dbw_globalmart"))
      3 display(
----> 4     spark.sql("SHOW VOLUMES IN dbw_globalmart.your_schema")
      5 )

File /databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/session.py:919, in SparkSession.sql(self, sqlQuery, args, **kwargs)
    916         _views.append(SubqueryAlias(df._plan, name))
    918 cmd = SQL(sqlQuery, _args, _named_args, _views)
--> 919 data, properties, ei = self.client.execute_command(cmd.command(self._client))
    920 if "sql_command_result" in properties:
    921     df = DataFrame(CachedRelation(properties["sql_command_result"]), self)

File /databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/client/core.py:1812, in SparkConnectClient.execute_command(self, command, observations, extra_request_m

In [0]:
display(
    spark.sql("SHOW VOLUMES IN dbw_globalmart.default")
)

database,volume_name


In [0]:
base_path = (
    "abfss://globalmart@globalmartpiyush.dfs.core.windows.net/"
    "gold/star_schema/"
)

df = spark.read.format("delta").load(
    base_path + "dim_product"
)

display(df)

product_key,product_id,product_name,category,subcategory,brand,unit_price,cost_price,created_date,updated_date
1,1001,Apex Mobile 1001,Electronics,Mobile,Zenith,37496,27030.64,2024-01-01,2025-01-01
2,1002,Nova Mobile 1002,Electronics,Mobile,Vertex,44233,32363.29,2024-01-01,2025-01-01
3,1003,Vertex Mobile 1003,Electronics,Mobile,Vertex,29744,16555.36,2024-01-01,2025-01-01
4,1004,Prime Laptop 1004,Electronics,Laptop,Apex,32392,25247.27,2024-01-01,2025-01-01
5,1005,Prime Laptop 1005,Electronics,Laptop,Prime,9845,7770.23,2024-01-01,2025-01-01
6,1006,Vertex Laptop 1006,Electronics,Laptop,Zenith,75754,55907.64,2024-01-01,2025-01-01
7,1007,Prime Audio 1007,Electronics,Audio,Vertex,65099,38398.27,2024-01-01,2025-01-01
8,1008,Nova Audio 1008,Electronics,Audio,Prime,1026,624.92,2024-01-01,2025-01-01
9,1009,Prime Audio 1009,Electronics,Audio,Nova,68189,47766.36,2024-01-01,2025-01-01
10,1010,Prime Accessories 1010,Electronics,Accessories,Vertex,16206,12419.99,2024-01-01,2025-01-01


In [0]:
base_path = (
    "abfss://globalmart@globalmartpiyush.dfs.core.windows.net/"
    "gold/star_schema/"
)

tables = [
    "dim_customer",
    "dim_product",
    "dim_store",
    "dim_date",
    "fact_sales"
]

for table_name in tables:
    print(f"===== {table_name} =====")

    df = spark.read.format("delta").load(
        base_path + table_name
    )

    print(f"Row count: {df.count()}")
    display(df)

===== dim_customer =====
Row count: 100


customer_key,customer_id,first_name,last_name,email,city,state,country,customer_segment,created_date,updated_date
1,1,Arjun,Sharma,arjun.sharma1@example.com,Indore,Madhya Pradesh,India,Consumer,2024-10-08,2024-12-09
2,2,Rahul,Nair,rahul.nair2@example.com,Mumbai,Maharashtra,India,Premium,2025-07-12,2025-08-03
3,3,Pooja,Sharma,pooja.sharma3@example.com,Pune,Maharashtra,India,Consumer,2024-04-05,2024-05-30
4,4,Nisha,Khan,nisha.khan4@example.com,Pune,Maharashtra,India,Premium,2025-07-28,2025-09-16
5,5,Riya,Kulkarni,riya.kulkarni5@example.com,Delhi,Delhi,India,Corporate,2025-04-04,2025-09-01
6,6,Aarav,Reddy,aarav.reddy6@example.com,Lucknow,Uttar Pradesh,India,Corporate,2024-06-12,2024-12-07
7,7,Ananya,Joshi,ananya.joshi7@example.com,Nagpur,Maharashtra,India,Consumer,2024-08-08,2024-11-02
8,8,Aditya,Kulkarni,aditya.kulkarni8@example.com,Mumbai,Maharashtra,India,Premium,2025-01-02,2025-03-31
9,9,Neha,Reddy,neha.reddy9@example.com,Pune,Maharashtra,India,Consumer,2025-04-15,2025-08-30
10,10,Sneha,Patil,sneha.patil10@example.com,Kolkata,West Bengal,India,Premium,2024-10-27,2025-04-05


===== dim_product =====
Row count: 60


product_key,product_id,product_name,category,subcategory,brand,unit_price,cost_price,created_date,updated_date
1,1001,Apex Mobile 1001,Electronics,Mobile,Zenith,37496,27030.64,2024-01-01,2025-01-01
2,1002,Nova Mobile 1002,Electronics,Mobile,Vertex,44233,32363.29,2024-01-01,2025-01-01
3,1003,Vertex Mobile 1003,Electronics,Mobile,Vertex,29744,16555.36,2024-01-01,2025-01-01
4,1004,Prime Laptop 1004,Electronics,Laptop,Apex,32392,25247.27,2024-01-01,2025-01-01
5,1005,Prime Laptop 1005,Electronics,Laptop,Prime,9845,7770.23,2024-01-01,2025-01-01
6,1006,Vertex Laptop 1006,Electronics,Laptop,Zenith,75754,55907.64,2024-01-01,2025-01-01
7,1007,Prime Audio 1007,Electronics,Audio,Vertex,65099,38398.27,2024-01-01,2025-01-01
8,1008,Nova Audio 1008,Electronics,Audio,Prime,1026,624.92,2024-01-01,2025-01-01
9,1009,Prime Audio 1009,Electronics,Audio,Nova,68189,47766.36,2024-01-01,2025-01-01
10,1010,Prime Accessories 1010,Electronics,Accessories,Vertex,16206,12419.99,2024-01-01,2025-01-01


===== dim_store =====
Row count: 15


store_key,store_id,store_name,city,state,country,region_id,region_name
1,101,GlobalMart Pune,Pune,Maharashtra,India,3,West
2,102,GlobalMart Mumbai,Mumbai,Maharashtra,India,3,West
3,103,GlobalMart Nagpur,Nagpur,Maharashtra,India,5,Central
4,104,GlobalMart Delhi,Delhi,Delhi,India,1,North
5,105,GlobalMart Jaipur,Jaipur,Rajasthan,India,1,North
6,106,GlobalMart Chennai,Chennai,Tamil Nadu,India,2,South
7,107,GlobalMart Bengaluru,Bengaluru,Karnataka,India,2,South
8,108,GlobalMart Hyderabad,Hyderabad,Telangana,India,2,South
9,109,GlobalMart Kolkata,Kolkata,West Bengal,India,4,East
10,110,GlobalMart Bhubaneswar,Bhubaneswar,Odisha,India,4,East


===== dim_date =====
Row count: 266


date_key,date,year,quarter,month,month_name,day,day_of_week,day_of_week_number,day_of_year,week_of_year,month_end_date
20250308,2025-03-08,2025,1,3,March,8,Saturday,7,67,10,2025-03-31
20250603,2025-06-03,2025,2,6,June,3,Tuesday,3,154,23,2025-06-30
20251223,2025-12-23,2025,4,12,December,23,Tuesday,3,357,52,2025-12-31
20251216,2025-12-16,2025,4,12,December,16,Tuesday,3,350,51,2025-12-31
20250406,2025-04-06,2025,2,4,April,6,Sunday,1,96,14,2025-04-30
20251222,2025-12-22,2025,4,12,December,22,Monday,2,356,52,2025-12-31
20250706,2025-07-06,2025,3,7,July,6,Sunday,1,187,27,2025-07-31
20251005,2025-10-05,2025,4,10,October,5,Sunday,1,278,40,2025-10-31
20250425,2025-04-25,2025,2,4,April,25,Friday,6,115,17,2025-04-30
20251012,2025-10-12,2025,4,10,October,12,Sunday,1,285,41,2025-10-31


===== fact_sales =====
Row count: 1298


sales_key_source_id,order_id,customer_key,product_key,store_key,date_key,quantity,unit_price,discount,gross_sales,net_sales
12,5006,96,2,9,20250611,3,44233.0,0.05,132699.0,132698.95
18,5009,69,48,3,20250108,4,48871.0,0.15,195484.0,195483.85
38,5017,13,42,13,20251109,2,15312.0,0.15,30624.0,30623.85
67,5027,74,56,10,20250428,5,49974.0,0.15,249870.0,249869.85
70,5030,43,27,6,20250628,1,57425.0,0.0,57425.0,57425.0
93,5039,35,59,7,20250918,4,36511.0,0.0,146044.0,146044.0
161,5063,51,24,5,20250112,1,46405.0,0.0,46405.0,46405.0
186,5073,96,52,11,20250708,2,35700.0,0.0,71400.0,71400.0
190,5074,14,46,11,20251123,3,39829.0,0.05,119487.0,119486.95
218,5087,85,39,14,20250331,1,62370.0,0.15,62370.0,62369.85
